In [4]:
import cv2
import mediapipe as mp
import numpy as np

def calculate_2d_angle(a, b, c):
    """Trigonometry for joint angles."""
    radians = np.arctan2(c[1] - b[1], c[0] - b[0]) - np.arctan2(a[1] - b[1], a[0] - b[0])
    angle = np.abs(radians * 180.0 / np.pi)
    if angle > 180.0: angle = 360.0 - angle
    return angle

# --- SETUP ---
mp_pose = mp.solutions.pose
# Use strict filter inside the live model
pose = mp_pose.Pose(min_detection_confidence=0.65, min_tracking_confidence=0.65)

# CHANGE THIS to your raw video path
video_path = "/Users/sofiasaifulrizal/Desktop/delta_2610/FYP2/zenodo/videos/Ex1/PM_109-Camera17-30fps.mp4"
output_path = "/Users/sofiasaifulrizal/Desktop/delta_2610/FYP2/Perfect_Dashboard_Demo.mp4"

cap = cv2.VideoCapture(video_path)

width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps = cap.get(cv2.CAP_PROP_FPS)

fourcc = cv2.VideoWriter_fourcc(*'avc1') # Mac compatible
out = cv2.VideoWriter(output_path, fourcc, fps, (width, height))

print("🎥 Rendering LIVE Kinematic Dashboard...")

while cap.isOpened():
    ret, frame = cap.read()
    if not ret: break

    # Process frame LIVE like FYP1
    image_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    results = pose.process(image_rgb)

    if results.pose_landmarks:
        landmarks = results.pose_landmarks.landmark
        
        # Grab the X, Y coordinates for the Right Arm
        shoulder = [landmarks[12].x * width, landmarks[12].y * height]
        elbow = [landmarks[14].x * width, landmarks[14].y * height]
        wrist = [landmarks[16].x * width, landmarks[16].y * height]
        
        # Check confidence (Visibility)
        if landmarks[12].visibility > 0.65 and landmarks[14].visibility > 0.65:
            
            # Calculate angle LIVE
            elbow_angle = calculate_2d_angle(shoulder, elbow, wrist)
            
            # Form Logic: Red or Green
            if elbow_angle >= 160.0:
                ui_color = (0, 255, 0) # Green
                status_text = f"Form: PASS ({elbow_angle:.1f} deg)"
            else:
                ui_color = (0, 0, 255) # Red
                status_text = f"FAIL: Keep Arm Straight! ({elbow_angle:.1f} deg)"

            # Draw the lines perfectly synced
            s_tup = (int(shoulder[0]), int(shoulder[1]))
            e_tup = (int(elbow[0]), int(elbow[1]))
            w_tup = (int(wrist[0]), int(wrist[1]))
            
            cv2.line(frame, s_tup, e_tup, ui_color, 4)
            cv2.line(frame, e_tup, w_tup, ui_color, 4)
            cv2.circle(frame, s_tup, 5, ui_color, -1)
            cv2.circle(frame, e_tup, 5, ui_color, -1)
            cv2.circle(frame, w_tup, 5, ui_color, -1)
            
            # Draw UI
            cv2.putText(frame, status_text, (40, 50), cv2.FONT_HERSHEY_SIMPLEX, 1, ui_color, 2)

    out.write(frame)

cap.release()
out.release()
print(f"✅ Video perfectly rendered to: {output_path}")

I0000 00:00:1777167575.969871       1 gl_context.cc:344] GL version: 2.1 (2.1 Metal - 90.5), renderer: Apple M1


🎥 Rendering LIVE Kinematic Dashboard...
✅ Video perfectly rendered to: /Users/sofiasaifulrizal/Desktop/delta_2610/FYP2/Perfect_Dashboard_Demo.mp4
